# Encrypt S3 objects with a Customer Managed KMS key and prove unauthorized roles cannot decrypt

Your compliance team passed an SOC 2 audit last year, but the auditor noted that S3 objects encrypted with the default AWS-managed key surface only generic `Decrypt` events in CloudTrail with no per-principal attribution. The auditor wants per-principal Decrypt visibility before the next review. You have one session to migrate the bucket to a Customer Managed Key, prove an authorized role can still read encrypted objects, and prove the denied role gets a hard `AccessDenied` at the KMS layer, not the S3 layer.

**Time.** Plan for about 50 minutes hands-on. KMS itself is fast; the cleanup at the end schedules the CMK for deletion with a 7-day waiting window (AWS does not allow hard deletes), so a fresh re-run of this lab within a week will see your previous key in `PendingDeletion` state and that is fine.

**Cost.** The lab itself costs less than a nickel: a fraction of a penny for the KMS key while it is alive in the session, a fraction of a penny for the S3 round trips. The bigger line item is the 23 cents you accrue over the 7-day waiting window after cleanup schedules the key for deletion. AWS does not let you hard-delete KMS keys for safety reasons; the 7-day minimum is the price of getting your audit trail back if you scheduled the wrong key by accident.

This lab maps to AWS SAA-C03 Domain 1: Design Secure Architectures (30% of exam weight).

In [ ]:
# NBVAL_SKIP
# Install labrun-checks. Pinned to a specific version per LAB-CREATION-BLUEPRINT.md
# build rules. Never use unpinned installs.

!pip install --quiet labrun-checks==0.5.0

In [ ]:
# NBVAL_SKIP
# Imports and per-lab constants. Resource names follow the
# labrun-{lab-slug}-{descriptor} pattern from LAB-CREATION-BLUEPRINT.md
# section 12.

import atexit
import getpass
import json
import time

import boto3
from botocore.exceptions import ClientError

from labrun_checks import (
    register,
    check,
    cleanup,
    run_cleanup,
    CleanupResource,
)

LAB_ID = "kms-encryption-and-key-policies"
LAB_TAG_KEY = "labrun:lab-slug"
LAB_TAG_VALUE = LAB_ID  # must equal cert YAML labs[1].slug exactly
REGION = "us-east-1"

AUTHORIZED_ROLE_NAME = f"labrun-{LAB_ID}-authorized-role"
DENIED_ROLE_NAME = f"labrun-{LAB_ID}-denied-role"
CMK_ALIAS = f"alias/labrun-{LAB_ID}"
SEED_KEY = "contracts/seed.txt"

BUCKET_NAME = None  # resolved after STS GetCallerIdentity
CMK_KEY_ID = None   # resolved after create_key in Task 1
CMK_KEY_ARN = None

In [ ]:
# NBVAL_SKIP
# Register the session, validate AWS credentials via STS GetCallerIdentity,
# and confirm the region. Per LAB-CREATION-BLUEPRINT section 15 this must
# succeed before the manifest cell creates anything.

session_token = getpass.getpass("Paste your session token from labrun.dev: ")
aws_access_key_id = getpass.getpass("AWS_ACCESS_KEY_ID: ")
aws_secret_access_key = getpass.getpass("AWS_SECRET_ACCESS_KEY: ")
aws_session_token = getpass.getpass(
    "AWS_SESSION_TOKEN (leave blank for long-lived credentials): "
).strip()
region_input = input(f"AWS region (default {REGION}): ").strip()
if region_input and region_input != REGION:
    print(f"Region {region_input!r} is not supported for this lab.")
    print(f"SAA Batch 1 labs run in {REGION} (N. Virginia).")
    print("Re-run this cell and accept the default.")
    raise SystemExit(1)

AWS_CREDENTIALS = {
    "aws_access_key_id": aws_access_key_id,
    "aws_secret_access_key": aws_secret_access_key,
    "region": REGION,
}
if aws_session_token:
    AWS_CREDENTIALS["aws_session_token"] = aws_session_token

sts = boto3.client(
    "sts",
    aws_access_key_id=aws_access_key_id,
    aws_secret_access_key=aws_secret_access_key,
    aws_session_token=aws_session_token or None,
    region_name=REGION,
)
try:
    identity = sts.get_caller_identity()
except ClientError as e:
    print("Credentials are missing or expired. STS GetCallerIdentity failed:")
    print(f"  {e}")
    print()
    print("Refresh your AWS credentials and re-run this cell.")
    raise SystemExit(1)

ACCOUNT_ID = identity["Account"]
CALLER_ARN = identity["Arn"]
print(f"Credentials validated. Account: {ACCOUNT_ID}")
print(f"Caller ARN: {CALLER_ARN}")
print(f"Region: {REGION}")
print(f"Session token in use: {bool(aws_session_token)}")

BUCKET_NAME = f"labrun-{LAB_ID}-{ACCOUNT_ID}"
print(f"Bucket name resolved: {BUCKET_NAME}")

register(session_token=session_token, lab_id=LAB_ID, credentials=AWS_CREDENTIALS)
print(f"Session registered for lab_id: {LAB_ID}")

In [ ]:
# NBVAL_SKIP
# Cleanup manifest, atexit safety net, and orphan scan.
# The KMS key is appended to CLEANUP_MANIFEST below AFTER create_key returns,
# because the key id is only known at that point. Cleanup schedules the key
# for deletion with the 7-day minimum waiting window. The describe dispatcher
# treats PendingDeletion as confirmed-gone so the tag audit does not flag
# the residual key during that window.

CLEANUP_MANIFEST: list[CleanupResource] = [
    CleanupResource(
        type="s3_bucket",
        id=BUCKET_NAME,
        region=REGION,
        cli_delete_command=f"aws s3 rb s3://{BUCKET_NAME} --force",
    ),
    CleanupResource(
        type="iam_role",
        id=AUTHORIZED_ROLE_NAME,
        region=REGION,
        cli_delete_command=f"aws iam delete-role --role-name {AUTHORIZED_ROLE_NAME}",
    ),
    CleanupResource(
        type="iam_role",
        id=DENIED_ROLE_NAME,
        region=REGION,
        cli_delete_command=f"aws iam delete-role --role-name {DENIED_ROLE_NAME}",
    ),
    CleanupResource(
        type="kms_alias",
        id=CMK_ALIAS,
        region=REGION,
        cli_delete_command=f"aws kms delete-alias --alias-name {CMK_ALIAS}",
    ),
]
# kms_key entry is appended in Task 1 after create_key returns the KeyId.


def _atexit_cleanup() -> None:
    try:
        run_cleanup(CLEANUP_MANIFEST)
    except Exception:
        pass


atexit.register(_atexit_cleanup)


def scan_for_orphans() -> list[str]:
    client = boto3.client(
        "resourcegroupstaggingapi",
        aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
        aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
        aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
        region_name=REGION,
    )
    arns: list[str] = []
    paginator = client.get_paginator("get_resources")
    for page in paginator.paginate(
        TagFilters=[{"Key": LAB_TAG_KEY, "Values": [LAB_TAG_VALUE]}],
    ):
        for item in page.get("ResourceTagMappingList", []):
            arns.append(item.get("ResourceARN", ""))
    return arns


orphans = scan_for_orphans()
if orphans:
    # KMS keys in PendingDeletion still appear in the tag audit until the
    # 7-day window elapses. Filter them out of the block so a student
    # re-running within a week is not stuck.
    kms_client = boto3.client(
        "kms",
        aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
        aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
        aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
        region_name=REGION,
    )
    active_orphans: list[str] = []
    for arn in orphans:
        if ":kms:" in arn and ":key/" in arn:
            try:
                desc = kms_client.describe_key(KeyId=arn)
                state = desc.get("KeyMetadata", {}).get("KeyState")
                if state == "PendingDeletion":
                    continue
            except ClientError:
                continue
        active_orphans.append(arn)
    if active_orphans:
        print(f"BLOCKED: existing resources tagged labrun:lab-slug={LAB_TAG_VALUE} were found:")
        for arn in active_orphans:
            print("  -", arn)
        print()
        print("These are leftovers from a previous run. Re-running setup")
        print("against an unclean state can produce duplicate resources.")
        print("Run the cleanup cell at the bottom of this notebook first,")
        print("then re-run setup.")
        raise SystemExit(1)

print("No active prior orphans found. Safe to create resources for this session.")

## Task 1: Create the CMK with an explicit key policy, alias, and lab tag

Two roles will exist in this lab: an authorized role that should be able to read the encrypted seed object, and a denied role that should fail at the KMS layer when it tries. The CMK key policy is what draws that boundary.

Build it in this order:

1. Create the two IAM roles up front with the account-root trust policy. The authorized role gets an inline policy granting `s3:GetObject` on `arn:aws:s3:::{bucket}/contracts/*`. The denied role gets the exact same S3 inline policy. This isolates the failure to the KMS layer in Task 4.
2. Create the CMK with an inline `Policy` JSON that names the lab caller ARN as the administrator (`kms:*`) AND names the authorized role ARN with `kms:Encrypt`, `kms:Decrypt`, `kms:GenerateDataKey`. The denied role is intentionally NOT in the key policy.
3. Create an alias pointing at the CMK so the rest of the lab can reference it by name.
4. Tag the CMK with `labrun:lab-slug=kms-encryption-and-key-policies`.

**Trap to avoid.** If you call `create_key` with NO `Policy` argument, AWS attaches a default policy granting only the AWS account root user `kms:*`. The lab caller cannot then administer the key. Always pass an explicit Policy at create time that names your caller ARN.

In [ ]:
# NBVAL_SKIP
# Task 1: Create authorized + denied roles, create the CMK with explicit key
# policy + alias + tag, append the kms_key entry to CLEANUP_MANIFEST.

iam = boto3.client(
    "iam",
    aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
    aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
    aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
    region_name=REGION,
)
kms = boto3.client(
    "kms",
    aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
    aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
    aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
    region_name=REGION,
)

trust_policy = {
    "Version": "2012-10-17",
    "Statement": [
        {
            "Effect": "Allow",
            "Principal": {"AWS": f"arn:aws:iam::{ACCOUNT_ID}:root"},
            "Action": "sts:AssumeRole",
        }
    ],
}
s3_read_inline = {
    "Version": "2012-10-17",
    "Statement": [
        {
            "Effect": "Allow",
            "Action": "s3:GetObject",
            "Resource": f"arn:aws:s3:::{BUCKET_NAME}/contracts/*",
        }
    ],
}

for role_name in (AUTHORIZED_ROLE_NAME, DENIED_ROLE_NAME):
    try:
        iam.create_role(
            RoleName=role_name,
            AssumeRolePolicyDocument=json.dumps(trust_policy),
            Description=f"labrun lab role {role_name}",
            Tags=[{"Key": LAB_TAG_KEY, "Value": LAB_TAG_VALUE}],
        )
        print(f"Created role: {role_name}")
    except ClientError as e:
        if e.response["Error"]["Code"] == "EntityAlreadyExists":
            print(f"Role {role_name} already exists, reusing.")
        else:
            raise
    iam.put_role_policy(
        RoleName=role_name,
        PolicyName=f"labrun-{role_name}-s3-read",
        PolicyDocument=json.dumps(s3_read_inline),
    )

AUTHORIZED_ROLE_ARN = f"arn:aws:iam::{ACCOUNT_ID}:role/{AUTHORIZED_ROLE_NAME}"
DENIED_ROLE_ARN = f"arn:aws:iam::{ACCOUNT_ID}:role/{DENIED_ROLE_NAME}"

key_policy = {
    "Version": "2012-10-17",
    "Statement": [
        {
            "Sid": "LabCallerAdmin",
            "Effect": "Allow",
            "Principal": {"AWS": CALLER_ARN},
            "Action": "kms:*",
            "Resource": "*",
        },
        {
            "Sid": "AccountRootAdmin",
            "Effect": "Allow",
            "Principal": {"AWS": f"arn:aws:iam::{ACCOUNT_ID}:root"},
            "Action": "kms:*",
            "Resource": "*",
        },
        {
            "Sid": "AuthorizedRoleEncryptDecrypt",
            "Effect": "Allow",
            "Principal": {"AWS": AUTHORIZED_ROLE_ARN},
            "Action": [
                "kms:Encrypt",
                "kms:Decrypt",
                "kms:GenerateDataKey",
                "kms:DescribeKey",
            ],
            "Resource": "*",
        },
    ],
}

# YOUR CODE: Create the symmetric Customer Managed Key using kms.create_key().
# Pass Description="labrun lab CMK", KeyUsage="ENCRYPT_DECRYPT",
# Policy=json.dumps(key_policy), and
# Tags=[{"TagKey": LAB_TAG_KEY, "TagValue": LAB_TAG_VALUE}]. Store the
# response in create_resp.

CMK_KEY_ID = create_resp["KeyMetadata"]["KeyId"]
CMK_KEY_ARN = create_resp["KeyMetadata"]["Arn"]
print(f"CMK created: {CMK_KEY_ID}")

# YOUR CODE: Create the alias using kms.create_alias() with
# AliasName=CMK_ALIAS and TargetKeyId=CMK_KEY_ID.

print(f"Alias created: {CMK_ALIAS} -> {CMK_KEY_ID}")

CLEANUP_MANIFEST.append(
    CleanupResource(
        type="kms_key",
        id=CMK_KEY_ID,
        region=REGION,
        cli_delete_command=(
            f"aws kms schedule-key-deletion --key-id {CMK_KEY_ID} --pending-window-in-days 7"
        ),
    )
)
print("CMK registered in CLEANUP_MANIFEST.")

In [ ]:
# NBVAL_SKIP
# Checkpoint 1: CMK exists with the lab tag, alias, and key policy granting
# the lab caller admin (kms:* or wildcard) and the authorized role at least
# kms:Encrypt + kms:Decrypt + kms:GenerateDataKey.

def checkpoint_1(session):
    from labrun_checks import CheckpointResult
    try:
        kms_client = boto3.client(
            "kms",
            aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
            aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
            aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
            region_name=REGION,
        )

        if not CMK_KEY_ID:
            return CheckpointResult(
                status="fail",
                error_reason="CMK_KEY_ID is not set. Run the Task 1 cell to create the key.",
            )

        desc = kms_client.describe_key(KeyId=CMK_KEY_ID)
        meta = desc.get("KeyMetadata", {})
        if meta.get("KeyState") != "Enabled":
            return CheckpointResult(
                status="fail",
                error_reason=f"Key {CMK_KEY_ID} state is {meta.get('KeyState')!r}, expected Enabled.",
            )
        if meta.get("KeyManager") != "CUSTOMER":
            return CheckpointResult(
                status="fail",
                error_reason=f"Key {CMK_KEY_ID} KeyManager is {meta.get('KeyManager')!r}, expected CUSTOMER.",
            )

        # Alias points at the key.
        alias_found = False
        paginator = kms_client.get_paginator("list_aliases")
        for page in paginator.paginate():
            for alias in page.get("Aliases", []):
                if alias.get("AliasName") == CMK_ALIAS and alias.get("TargetKeyId") == CMK_KEY_ID:
                    alias_found = True
                    break
            if alias_found:
                break
        if not alias_found:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"Alias {CMK_ALIAS} is not pointing at key {CMK_KEY_ID}. "
                    f"Confirm the create_alias call ran with TargetKeyId=CMK_KEY_ID."
                ),
            )

        # Lab tag.
        try:
            tag_resp = kms_client.list_resource_tags(KeyId=CMK_KEY_ID)
        except ClientError as e:
            return CheckpointResult(status="error", error_reason=str(e))
        tags = {t["TagKey"]: t["TagValue"] for t in tag_resp.get("Tags", [])}
        if tags.get(LAB_TAG_KEY) != LAB_TAG_VALUE:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"CMK is missing tag {LAB_TAG_KEY}={LAB_TAG_VALUE}. Found tags: {tags}"
                ),
            )

        # Key policy: admin clause for the caller + Encrypt/Decrypt clause for
        # the authorized role. Accept wildcard actions (kms:*, *).
        pol = kms_client.get_key_policy(KeyId=CMK_KEY_ID, PolicyName="default")
        policy_doc = json.loads(pol["Policy"])
        statements = policy_doc.get("Statement", [])

        action_wildcards = {"kms:*", "*"}
        required_auth_actions = {"kms:Encrypt", "kms:Decrypt", "kms:GenerateDataKey"}

        caller_admin_ok = False
        authorized_role_ok = False
        for stmt in statements:
            if stmt.get("Effect") != "Allow":
                continue
            p = stmt.get("Principal", {})
            aws_principal = p.get("AWS")
            if isinstance(aws_principal, str):
                principals = [aws_principal]
            elif isinstance(aws_principal, list):
                principals = aws_principal
            else:
                principals = []
            actions = stmt.get("Action", [])
            if isinstance(actions, str):
                actions = [actions]
            action_set = set(actions)

            covers_admin = bool(action_set & action_wildcards)
            account_root_arn = f"arn:aws:iam::{ACCOUNT_ID}:root"
            if covers_admin and (CALLER_ARN in principals or account_root_arn in principals):
                caller_admin_ok = True

            authorized_role_arn = f"arn:aws:iam::{ACCOUNT_ID}:role/{AUTHORIZED_ROLE_NAME}"
            if authorized_role_arn in principals:
                if required_auth_actions.issubset(action_set) or bool(action_set & action_wildcards):
                    authorized_role_ok = True

        if not caller_admin_ok:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"Key policy is missing an Allow statement that names the lab caller "
                    f"({CALLER_ARN}) or account root with kms:* admin permissions."
                ),
            )
        if not authorized_role_ok:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"Key policy is missing an Allow statement that names "
                    f"{AUTHORIZED_ROLE_NAME} as Principal and grants at least "
                    f"kms:Encrypt, kms:Decrypt, and kms:GenerateDataKey (or a wildcard covering them)."
                ),
            )

        return CheckpointResult(status="pass")
    except Exception as e:
        return CheckpointResult(status="error", error_reason=str(e))


check(1, checkpoint_1)

<details><summary>Hint 1 (nudge)</summary>

The checkpoint walks the key policy looking for two specific Allow statements. The failure message tells you which one is missing: the admin clause for your caller or the Encrypt/Decrypt clause for the authorized role.

</details>

<details><summary>Hint 2 (direction)</summary>

The key policy is a JSON document you pass to `kms.create_key(Policy=...)`. Two Allow statements are required: one with `Principal.AWS` set to `CALLER_ARN` (or account root) and `Action: kms:*`, and a second with `Principal.AWS` set to `AUTHORIZED_ROLE_ARN` and `Action` listing at least `kms:Encrypt`, `kms:Decrypt`, and `kms:GenerateDataKey`. If you forget the Policy parameter entirely, AWS attaches a default policy that locks the lab caller out.

</details>

<details><summary>Hint 3 (spoiler)</summary>

Complete working solution for Task 1 (the create_key and create_alias lines that were YOUR CODE):

```python
create_resp = kms.create_key(
    Description="labrun lab CMK",
    KeyUsage="ENCRYPT_DECRYPT",
    Policy=json.dumps(key_policy),
    Tags=[{"TagKey": LAB_TAG_KEY, "TagValue": LAB_TAG_VALUE}],
)
CMK_KEY_ID = create_resp["KeyMetadata"]["KeyId"]
CMK_KEY_ARN = create_resp["KeyMetadata"]["Arn"]

kms.create_alias(AliasName=CMK_ALIAS, TargetKeyId=CMK_KEY_ID)
```

</details>

**Wallet check.** Costs so far: fractions of a penny. KMS CMKs are billed at about $1/month prorated for their existence; a 50-minute session is about $0.001. The 7-day scheduled-deletion residual after cleanup is the bigger line item at about $0.23 per key.

## Task 2: Create the bucket with SSE-KMS default encryption pointing at the CMK

Bucket default encryption is the durable way to ensure every object is encrypted under the CMK without having to set per-PUT headers. Once the bucket is wired this way, every `put_object` call (whether you remembered to ask for encryption or not) lands as SSE-KMS under the lab CMK.

Build it in this order:

1. Create the bucket and tag it with `labrun:lab-slug=kms-encryption-and-key-policies`.
2. Call `s3.put_bucket_encryption` with a `ServerSideEncryptionConfiguration` whose first rule sets `SSEAlgorithm=aws:kms` and `KMSMasterKeyID` to the CMK ARN.
3. Upload one seed object under `contracts/seed.txt`. No `ServerSideEncryption` headers needed; the bucket default applies.

In [ ]:
# NBVAL_SKIP
# Task 2: Create bucket, set SSE-KMS default encryption pointing at the CMK,
# upload the seed object.

s3 = boto3.client(
    "s3",
    aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
    aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
    aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
    region_name=REGION,
)

# YOUR CODE: Create the S3 bucket using s3.create_bucket(Bucket=BUCKET_NAME).
# us-east-1 rejects LocationConstraint; other regions require
# CreateBucketConfiguration={"LocationConstraint": REGION}.

s3.put_bucket_tagging(
    Bucket=BUCKET_NAME,
    Tagging={"TagSet": [{"Key": LAB_TAG_KEY, "Value": LAB_TAG_VALUE}]},
)
print(f"Bucket created and tagged: {BUCKET_NAME}")

encryption_config = {
    "Rules": [
        {
            "ApplyServerSideEncryptionByDefault": {
                "SSEAlgorithm": "aws:kms",
                "KMSMasterKeyID": CMK_KEY_ARN,
            },
            "BucketKeyEnabled": True,
        }
    ]
}
# YOUR CODE: Apply the encryption config using s3.put_bucket_encryption() with
# Bucket=BUCKET_NAME and ServerSideEncryptionConfiguration=encryption_config.

print(f"Default encryption set: SSE-KMS pointing at {CMK_KEY_ARN}")

# YOUR CODE: Upload the seed object using s3.put_object() with
# Bucket=BUCKET_NAME, Key=SEED_KEY, and Body=b"contract revenue Q4 2026\n".

print(f"Seed object uploaded: s3://{BUCKET_NAME}/{SEED_KEY}")

In [ ]:
# NBVAL_SKIP
# Checkpoint 2: Bucket default encryption is SSE-KMS pointing at the lab CMK ARN.

def checkpoint_2(session):
    from labrun_checks import CheckpointResult
    try:
        s3_client = boto3.client(
            "s3",
            aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
            aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
            aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
            region_name=REGION,
        )
        try:
            enc = s3_client.get_bucket_encryption(Bucket=BUCKET_NAME)
        except ClientError as e:
            code = e.response["Error"]["Code"]
            if code in ("ServerSideEncryptionConfigurationNotFoundError", "NoSuchBucket"):
                return CheckpointResult(
                    status="fail",
                    error_reason=(
                        f"Bucket {BUCKET_NAME} has no default encryption configured. "
                        f"Call put_bucket_encryption with SSE-KMS."
                    ),
                )
            return CheckpointResult(status="error", error_reason=str(e))

        rules = enc.get("ServerSideEncryptionConfiguration", {}).get("Rules", [])
        if not rules:
            return CheckpointResult(
                status="fail",
                error_reason="Bucket encryption configuration has no Rules.",
            )
        default = rules[0].get("ApplyServerSideEncryptionByDefault", {})
        algo = default.get("SSEAlgorithm")
        if algo != "aws:kms":
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"SSEAlgorithm is {algo!r}, expected 'aws:kms'. SSE-S3 or SSE-C "
                    f"do not satisfy the CMK requirement."
                ),
            )
        key_id_field = default.get("KMSMasterKeyID", "")
        # Accept either the bare KeyId or the full ARN form.
        if CMK_KEY_ID not in key_id_field and (CMK_KEY_ARN or "") != key_id_field:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"Bucket KMSMasterKeyID is {key_id_field!r}, expected to reference "
                    f"the lab CMK ({CMK_KEY_ARN})."
                ),
            )
        return CheckpointResult(status="pass")
    except Exception as e:
        return CheckpointResult(status="error", error_reason=str(e))


check(2, checkpoint_2)

<details><summary>Hint 1 (nudge)</summary>

The checkpoint failure says either the encryption config is missing entirely or the algorithm is not `aws:kms` or the key ID does not point at your CMK. Read the message to know which.

</details>

<details><summary>Hint 2 (direction)</summary>

The shape: `s3.put_bucket_encryption(Bucket=BUCKET_NAME, ServerSideEncryptionConfiguration={"Rules": [{"ApplyServerSideEncryptionByDefault": {"SSEAlgorithm": "aws:kms", "KMSMasterKeyID": <full CMK ARN>}, "BucketKeyEnabled": True}]})`. Pass `CMK_KEY_ARN`, not the alias and not the bare key ID.

</details>

<details><summary>Hint 3 (spoiler)</summary>

Complete working solution for Task 2:

```python
s3 = boto3.client(
    "s3",
    aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
    aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
    aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
    region_name=REGION,
)

if REGION == "us-east-1":
    s3.create_bucket(Bucket=BUCKET_NAME)
else:
    s3.create_bucket(
        Bucket=BUCKET_NAME,
        CreateBucketConfiguration={"LocationConstraint": REGION},
    )

s3.put_bucket_tagging(
    Bucket=BUCKET_NAME,
    Tagging={"TagSet": [{"Key": LAB_TAG_KEY, "Value": LAB_TAG_VALUE}]},
)

encryption_config = {
    "Rules": [
        {
            "ApplyServerSideEncryptionByDefault": {
                "SSEAlgorithm": "aws:kms",
                "KMSMasterKeyID": CMK_KEY_ARN,
            },
            "BucketKeyEnabled": True,
        }
    ]
}
s3.put_bucket_encryption(
    Bucket=BUCKET_NAME,
    ServerSideEncryptionConfiguration=encryption_config,
)

s3.put_object(Bucket=BUCKET_NAME, Key=SEED_KEY, Body=b"contract revenue Q4 2026\n")
```

</details>

**Wallet check.** A handful of S3 PUTs against a tiny object and one GenerateDataKey call still rounds to fractions of a penny. KMS gives you 20,000 free requests per month.

## Task 3: Authorized role reads the encrypted object end to end

The authorized role has S3 GetObject permission AND a key-policy clause granting `kms:Decrypt`. The S3 GetObject path internally calls KMS Decrypt to unseal the object before returning the plaintext. If either layer is missing, the call fails.

Build it in this order:

1. Assume the authorized role.
2. Build a fresh S3 client from the temp credentials (with `aws_session_token`).
3. Call `get_object` on `contracts/seed.txt` and confirm the response `ServerSideEncryption` is `aws:kms` and `SSEKMSKeyId` matches the lab CMK ARN.

In [ ]:
# NBVAL_SKIP
# Task 3: Assume the authorized role and read the encrypted seed object.

sts = boto3.client(
    "sts",
    aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
    aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
    aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
    region_name=REGION,
)

AUTHORIZED_ROLE_ARN = f"arn:aws:iam::{ACCOUNT_ID}:role/{AUTHORIZED_ROLE_NAME}"

# YOUR CODE: Call sts.assume_role(RoleArn=AUTHORIZED_ROLE_ARN,
# RoleSessionName="labrun-authorized-read") and store the response in
# assume_resp.

temp_creds = assume_resp["Credentials"]

# YOUR CODE: Build a fresh boto3 S3 client from temp_creds with
# aws_access_key_id=temp_creds["AccessKeyId"],
# aws_secret_access_key=temp_creds["SecretAccessKey"],
# aws_session_token=temp_creds["SessionToken"], region_name=REGION.
# Assign it to s3_authorized.

resp = s3_authorized.get_object(Bucket=BUCKET_NAME, Key=SEED_KEY)
print("GetObject succeeded as authorized role.")
print(f"  ServerSideEncryption: {resp.get('ServerSideEncryption')}")
print(f"  SSEKMSKeyId:          {resp.get('SSEKMSKeyId')}")
body = resp["Body"].read()
print(f"  Body bytes:           {body!r}")

In [ ]:
# NBVAL_SKIP
# Checkpoint 3: Independently assume the authorized role, build a fresh
# client, and confirm GetObject returns 200 with SSE-KMS metadata pointing
# at the lab CMK.

def checkpoint_3(session):
    from labrun_checks import CheckpointResult
    try:
        sts_client = boto3.client(
            "sts",
            aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
            aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
            aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
            region_name=REGION,
        )
        role_arn = f"arn:aws:iam::{ACCOUNT_ID}:role/{AUTHORIZED_ROLE_NAME}"

        last_error = None
        delay = 2
        elapsed = 0
        assume_resp = None
        while elapsed < 60:
            try:
                assume_resp = sts_client.assume_role(
                    RoleArn=role_arn,
                    RoleSessionName="labrun-checkpoint-3",
                )
                last_error = None
                break
            except ClientError as e:
                last_error = e
                time.sleep(delay)
                elapsed += delay
                delay = min(delay * 2, 8)
        if last_error is not None or assume_resp is None:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"Could not assume {role_arn} within 60s. "
                    f"{repr(last_error) if last_error else 'no response'}"
                ),
            )

        temp_creds = assume_resp["Credentials"]
        s3_assumed = boto3.client(
            "s3",
            aws_access_key_id=temp_creds["AccessKeyId"],
            aws_secret_access_key=temp_creds["SecretAccessKey"],
            aws_session_token=temp_creds["SessionToken"],
            region_name=REGION,
        )
        try:
            resp = s3_assumed.get_object(Bucket=BUCKET_NAME, Key=SEED_KEY)
        except ClientError as e:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"Authorized role GetObject failed: "
                    f"{e.response['Error']['Code']}: {e.response['Error']['Message']}. "
                    f"Check the key policy clause for {AUTHORIZED_ROLE_NAME}."
                ),
            )
        if resp.get("ServerSideEncryption") != "aws:kms":
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"Response ServerSideEncryption is {resp.get('ServerSideEncryption')!r}, "
                    f"expected 'aws:kms'. Bucket default encryption may not be applied."
                ),
            )
        sse_key = resp.get("SSEKMSKeyId", "")
        if CMK_KEY_ID not in sse_key and (CMK_KEY_ARN or "") != sse_key:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"Response SSEKMSKeyId is {sse_key!r}, expected to reference "
                    f"the lab CMK ({CMK_KEY_ARN}). Bucket default encryption is wired "
                    f"to a different key."
                ),
            )
        return CheckpointResult(status="pass")
    except Exception as e:
        return CheckpointResult(status="error", error_reason=str(e))


check(3, checkpoint_3)

<details><summary>Hint 1 (nudge)</summary>

Two failures are possible. Either the role cannot assume (trust policy) or the role assumes but GetObject fails (S3 IAM, bucket policy, or KMS Decrypt missing). The checkpoint message tells you which.

</details>

<details><summary>Hint 2 (direction)</summary>

Build a NEW boto3 S3 client from the temp credentials. Pass `aws_session_token=temp_creds["SessionToken"]`. If you reuse your base `s3` client, you are testing your own permissions instead of the role's. If assume_role itself fails repeatedly, IAM has not propagated yet (the validator retries for 60 seconds) or the trust policy Principal does not match your caller account.

</details>

<details><summary>Hint 3 (spoiler)</summary>

Complete working solution for Task 3:

```python
sts = boto3.client(
    "sts",
    aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
    aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
    aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
    region_name=REGION,
)
AUTHORIZED_ROLE_ARN = f"arn:aws:iam::{ACCOUNT_ID}:role/{AUTHORIZED_ROLE_NAME}"

assume_resp = sts.assume_role(
    RoleArn=AUTHORIZED_ROLE_ARN,
    RoleSessionName="labrun-authorized-read",
)
temp_creds = assume_resp["Credentials"]

s3_authorized = boto3.client(
    "s3",
    aws_access_key_id=temp_creds["AccessKeyId"],
    aws_secret_access_key=temp_creds["SecretAccessKey"],
    aws_session_token=temp_creds["SessionToken"],
    region_name=REGION,
)
resp = s3_authorized.get_object(Bucket=BUCKET_NAME, Key=SEED_KEY)
```

</details>

**Wallet check.** Still under a penny. STS AssumeRole is free, GetObject on a tiny object is fractions of a cent, KMS Decrypt is one request against your 20k monthly free tier.

## Task 4: Denied role gets AccessDenied at the KMS layer

The denied role has the same S3 GetObject IAM permission as the authorized role. The bucket policy does not block it. The failure happens at the KMS layer: the GetObject path tries to call Decrypt against the CMK, the key policy does not grant `kms:Decrypt` to the denied role, and the call fails. From the student's perspective the error looks like a generic AccessDenied; from CloudTrail's perspective it is a Decrypt event with `errorCode: AccessDenied`.

Build it in this order:

1. Assume the denied role.
2. Build a fresh S3 client from the temp credentials.
3. Call `get_object` on `contracts/seed.txt` and assert that it raises `ClientError` with `Error.Code == "AccessDenied"`.

In [ ]:
# NBVAL_SKIP
# Task 4: Assume the denied role, fresh client, expect AccessDenied at the KMS layer.

sts = boto3.client(
    "sts",
    aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
    aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
    aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
    region_name=REGION,
)

DENIED_ROLE_ARN = f"arn:aws:iam::{ACCOUNT_ID}:role/{DENIED_ROLE_NAME}"

# YOUR CODE: Call sts.assume_role(RoleArn=DENIED_ROLE_ARN,
# RoleSessionName="labrun-denied-read") and store the response in assume_resp.

temp_creds = assume_resp["Credentials"]

# YOUR CODE: Build a fresh boto3 S3 client from temp_creds with
# aws_session_token=temp_creds["SessionToken"] and assign it to s3_denied.

try:
    s3_denied.get_object(Bucket=BUCKET_NAME, Key=SEED_KEY)
    print("UNEXPECTED: denied role read the object. Check the key policy.")
except ClientError as e:
    code_str = e.response["Error"]["Code"]
    print(f"Denied as expected: {code_str}")
    print(f"  Full message: {e.response['Error']['Message']}")

In [ ]:
# NBVAL_SKIP
# Checkpoint 4: Denied role gets AccessDenied on GetObject for the same
# SSE-KMS object. The denial happens at the KMS layer because the bucket
# policy and the role's IAM both allow S3 GetObject; the key policy is the
# only surface that blocks the denied role.

def checkpoint_4(session):
    from labrun_checks import CheckpointResult
    try:
        sts_client = boto3.client(
            "sts",
            aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
            aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
            aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
            region_name=REGION,
        )
        role_arn = f"arn:aws:iam::{ACCOUNT_ID}:role/{DENIED_ROLE_NAME}"

        last_error = None
        delay = 2
        elapsed = 0
        assume_resp = None
        while elapsed < 60:
            try:
                assume_resp = sts_client.assume_role(
                    RoleArn=role_arn,
                    RoleSessionName="labrun-checkpoint-4",
                )
                last_error = None
                break
            except ClientError as e:
                last_error = e
                time.sleep(delay)
                elapsed += delay
                delay = min(delay * 2, 8)
        if last_error is not None or assume_resp is None:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"Could not assume {role_arn} within 60s. "
                    f"{repr(last_error) if last_error else 'no response'}"
                ),
            )

        temp_creds = assume_resp["Credentials"]
        s3_assumed = boto3.client(
            "s3",
            aws_access_key_id=temp_creds["AccessKeyId"],
            aws_secret_access_key=temp_creds["SecretAccessKey"],
            aws_session_token=temp_creds["SessionToken"],
            region_name=REGION,
        )
        try:
            s3_assumed.get_object(Bucket=BUCKET_NAME, Key=SEED_KEY)
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"Denied role unexpectedly read the encrypted object. The key "
                    f"policy is granting {DENIED_ROLE_NAME} Decrypt access."
                ),
            )
        except ClientError as e:
            code_str = e.response["Error"]["Code"]
            if code_str != "AccessDenied":
                return CheckpointResult(
                    status="fail",
                    error_reason=(
                        f"Expected AccessDenied, got {code_str}. "
                        f"{e.response['Error']['Message']}"
                    ),
                )
        return CheckpointResult(status="pass")
    except Exception as e:
        return CheckpointResult(status="error", error_reason=str(e))


check(4, checkpoint_4)

<details><summary>Hint 1 (nudge)</summary>

If the denied role unexpectedly succeeded, the key policy has an Allow statement that names the denied role (or a too-broad principal). If the call fails with the wrong error code, the failure is happening at the S3 layer instead of the KMS layer.

</details>

<details><summary>Hint 2 (direction)</summary>

The denied role's inline policy needs `s3:GetObject` on the bucket so the request reaches S3 authorization. The bucket has no bucket policy denying the role. The KMS key policy must NOT include a statement for the denied role. The expected error code is `AccessDenied` with a message that often references KMS Decrypt.

</details>

<details><summary>Hint 3 (spoiler)</summary>

Complete working solution for Task 4:

```python
sts = boto3.client(
    "sts",
    aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
    aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
    aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
    region_name=REGION,
)
DENIED_ROLE_ARN = f"arn:aws:iam::{ACCOUNT_ID}:role/{DENIED_ROLE_NAME}"

assume_resp = sts.assume_role(
    RoleArn=DENIED_ROLE_ARN,
    RoleSessionName="labrun-denied-read",
)
temp_creds = assume_resp["Credentials"]

s3_denied = boto3.client(
    "s3",
    aws_access_key_id=temp_creds["AccessKeyId"],
    aws_secret_access_key=temp_creds["SecretAccessKey"],
    aws_session_token=temp_creds["SessionToken"],
    region_name=REGION,
)
try:
    s3_denied.get_object(Bucket=BUCKET_NAME, Key=SEED_KEY)
except ClientError as e:
    assert e.response["Error"]["Code"] == "AccessDenied"
```

</details>

**Wallet check.** Still under a penny in-session. One STS assume_role, one denied GetObject attempt (the KMS Decrypt call that fires before AccessDenied is still a billable Decrypt request, well inside your 20k monthly free tier).

## Cleanup

Time to tear it all down. The cell below schedules the CMK for deletion with a 7-day waiting window, then deletes the alias, both roles, and the bucket. Running cleanup twice is safe; the second run will see the key in `PendingDeletion` and treat that as already-gone.

In [ ]:
# NBVAL_SKIP
# Cleanup. Tear down every resource in CLEANUP_MANIFEST. KMS keys cannot be
# hard-deleted; the kms_key adapter schedules deletion with PendingWindowInDays=7
# and the describe path treats PendingDeletion as confirmed-gone.

import sys

result = run_cleanup(CLEANUP_MANIFEST)

for warning in result.warnings:
    print(warning)
for orphan in result.orphans:
    print(orphan)
if result.warnings or result.orphans:
    print()

failed_ids: set[str] = set()
for w in result.warnings:
    for r in CLEANUP_MANIFEST:
        if r.id in w:
            failed_ids.add(r.id)
            break

critical_total = 0
standard_total = len(CLEANUP_MANIFEST)
critical_destroyed = critical_total
standard_destroyed = standard_total - len(failed_ids)
failed_count = len(failed_ids)

print("Cleanup complete.")
print(f"Critical resources destroyed: {critical_destroyed}")
print(f"Standard resources destroyed: {standard_destroyed}")
print(f"Resources that failed to delete: {failed_count} (see above for CLI commands)")
print("If K > 0, your AWS account may still incur charges. Resolve before closing this notebook.")

cleanup(status=result.status)

if failed_count > 0:
    sys.exit(1)

**Session total: under $0.05 in-session, plus approximately $0.23 residual per CMK over the 7-day scheduled-deletion window.** AWS does not let you hard-delete KMS keys for safety reasons; running this lab twice in a week leaves two pending-deletion keys, each accruing the residual until its window elapses.

## Reflection

These are not graded. They are for you.

1. Why does AWS impose a minimum 7-day waiting period on `schedule_key_deletion` instead of letting you hard-delete a CMK immediately? What operational risk does the waiting window mitigate, and why is hard-deletion not a feature even for keys with no ciphertexts in the wild?

2. Walk through every policy surface AWS evaluates when an IAM role calls `s3:GetObject` on an SSE-KMS-encrypted object. List each evaluation step in order and what permission each surface must grant for the call to succeed.

## Exam-Style Practice

**Question 1 (Domain 1, encryption key management for audit):**

A compliance team requires that every Decrypt call against a sensitive S3 bucket is logged in CloudTrail with the specific IAM principal that performed it. The team is choosing between three S3 encryption options. Which option satisfies the requirement?

A. SSE-S3 (Amazon S3-managed keys), because S3 logs every decryption in CloudTrail by default.

B. SSE-KMS with the AWS-managed `aws/s3` key, because the AWS-managed key logs every Decrypt in CloudTrail with the calling principal.

C. SSE-KMS with a Customer Managed Key (CMK), because Decrypt calls against a CMK surface in CloudTrail with the calling IAM principal.

D. SSE-C (customer-provided keys), because the customer key material is included in the CloudTrail event.

<details><summary>Show answer</summary>

**Correct: C.**

- A is wrong: SSE-S3 keys are managed entirely by S3 and Decrypt operations do not surface in CloudTrail at all.
- B is partially right that CloudTrail logs Decrypt events for KMS, but the AWS-managed `aws/s3` key has a generic key policy that does not always attribute calls to the original IAM principal in a way that satisfies SOC 2 audit asks; the standard guidance for per-principal Decrypt attribution is a CMK.
- C is correct: a Customer Managed Key produces CloudTrail `Decrypt` events that include the calling IAM principal, satisfying the per-principal audit requirement.
- D is wrong: SSE-C keeps the key material with the customer; AWS never receives or logs the key, and there is no audit trail of decryption events on the AWS side.

</details>

**Question 2 (Domain 1, delegating Decrypt to another principal):**

You need to allow a third IAM role to decrypt objects encrypted with a CMK you own. The role is in the same AWS account. You want the delegation to be revocable, scoped to a specific time window, and visible to the security team without modifying the CMK's primary key policy. Which mechanism best fits?

A. Add a Statement to the CMK's key policy granting the role `kms:Decrypt`, then remove the Statement when the time window ends.

B. Attach an IAM policy to the role granting `kms:Decrypt` on the CMK ARN, then detach the policy when the window ends.

C. Create a KMS grant on the CMK with `GranteePrincipal` set to the role and `Operations` set to `Decrypt`, then revoke the grant when the window ends.

D. Add the role to the CMK's `via-service` Condition for `kms.amazonaws.com`, then remove it when the window ends.

<details><summary>Show answer</summary>

**Correct: C.**

- A works but modifying the key policy directly is heavyweight, requires admin permissions on the CMK, and grants permanently until you edit the policy back.
- B works but IAM policies grant access without a clear revocation signal on the CMK side, and the security team has to inspect the role's attached policies rather than the CMK to audit who can Decrypt.
- C is correct: KMS grants are the AWS-designed mechanism for time-bounded, scoped, revocable delegation. `revoke_grant` is the explicit lifecycle, and grants appear in `kms.list_grants` for the security team to audit alongside the key policy.
- D is wrong: `kms:via-service` is a Condition that scopes the calling AWS service (e.g., S3, EBS), not a way to add principals.

</details>